# N13 — Safety Car Probability: EDA & Labeling

The goal of this notebook is to build the labeled dataset for the safety car probability
model (N14). We work at the level of **race laps**: one row per (race, lap_number),
aggregating the observable state of the field at that point in the race.

For each lap we determine whether a Safety Car or Virtual Safety Car deployment
occurred within the next 5 laps — defining the binary `sc_within_5_laps` label.

The output is a Parquet file with race-state features and a binary SC label,
ready for Random Forest training in N14.

> **Dataset note:** unlike the overtake dataset (~28k pairs), the SC dataset is
> significantly smaller — ~3,500 rows (58 races × ~60 laps). Class imbalance
> and validation strategy are designed accordingly.

**Data sources:**
- `session.laps` — lap times, tyre data, positions, retirements (2023–2025)
- `session.track_status` — timestamped track status changes at second-level resolution;
  used to derive yellow flag counts and escalation patterns per lap window
- `session.race_control_messages` — timestamped FIA messages (incidents, yellow sectors,
  SC/VSC announcements); captures the precursor signals that precede SC deployments
- `data/processed/circuit_clustering/` — circuit cluster assignments

**Exports:**
- EDA plots → `notebooks/strategy/sc_probability/outputs/`
- Labeled dataset → `data/processed/sc_labeled/`


---

## Step 0 - Setup and Imports 

In [1]:
# ── Step 0 — Setup ────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")
import pathlib
import json
import fastf1
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

# ── Repo root ─────────────────────────────────────────────────────────────────
repo_root = pathlib.Path.cwd()
while not (repo_root / ".git").exists():
    repo_root = repo_root.parent

# ── Paths ─────────────────────────────────────────────────────────────────────
OUTPUTS    = repo_root / "notebooks" / "strategy" / "sc_probability" / "outputs"
PROCESSED  = repo_root / "data" / "processed" / "sc_labeled"
CACHE      = repo_root / "data" / "cache" / "fastf1"
CLUSTERING = repo_root / "data" / "processed" / "circuit_clustering"

OUTPUTS.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

fastf1.Cache.enable_cache(str(CACHE))


In [2]:
print(f"Repo root  : {repo_root}")
print(f"Outputs    : {OUTPUTS}")
print(f"Processed  : {PROCESSED}")
print(f"Clustering : {CLUSTERING}")
print(f"FF1 cache  : {CACHE}")

Repo root  : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager
Outputs    : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\notebooks\strategy\sc_probability\outputs
Processed  : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\processed\sc_labeled
Clustering : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\processed\circuit_clustering
FF1 cache  : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\cache\fastf1


---

## Step 1 — Schedule & Circuit Clusters

Load the race calendar for 2023–2025 and attach the circuit cluster label to each
event. The cluster (0–3) encodes the circuit type (power, high-downforce, balanced,
street) and will be used as a categorical feature in N14.

**Circuit cluster source:** `data/processed/circuit_clustering/circuit_clusters_k4.parquet`
— produced in the circuit clustering notebook and reused across all strategy models.


In [3]:
# ── Step 1 — Load schedule + circuit clusters ─────────────────────────────────
SEASONS = [2023, 2024, 2025]

# ── Circuit cluster mapping ────────────────────────────────────────────────────
_clusters_df = pd.read_parquet(CLUSTERING / "circuit_clusters_k4.parquet")
cluster_map = dict(zip(_clusters_df["GP_Name"], _clusters_df["Cluster"]))

print(f"Circuit clusters loaded: {len(cluster_map)} circuits")
print(cluster_map)

# ── Race schedule (exclude testing, sprints count as separate events) ──────────
schedule_rows = []
for year in SEASONS:
    sched = fastf1.get_event_schedule(year, include_testing=False)
    races = sched[sched["EventFormat"].isin(["conventional", "sprint_shootout", "sprint"])]
    # Keep only rounds with a Race session
    races = races[races["Session5"] == "Race"].copy()
    races["year"] = year
    schedule_rows.append(races[["year", "RoundNumber", "EventName", "Location", "Country"]])

schedule = pd.concat(schedule_rows, ignore_index=True)

# ── Attach circuit cluster ─────────────────────────────────────────────────────
def resolve_cluster(location: str) -> int:
    for key, cluster in cluster_map.items():
        if key.lower() in location.lower() or location.lower() in key.lower():
            return cluster
    return -1   # unknown

schedule["circuit_cluster"] = schedule["Location"].apply(resolve_cluster)

unknown = schedule[schedule["circuit_cluster"] == -1]
print(f"\nSchedule loaded: {len(schedule)} races across {SEASONS}")
print(f"Unknown clusters: {len(unknown)}")
if len(unknown):
    print(unknown[["year", "EventName", "Location"]].to_string(index=False))

print(f"\nCluster distribution:\n{schedule['circuit_cluster'].value_counts().sort_index()}")


Circuit clusters loaded: 25 circuits
{'Austin': 1, 'Baku': 1, 'Barcelona': 3, 'Budapest': 3, 'Imola': 1, 'Jeddah': 1, 'Las Vegas': 0, 'Lusail': 0, 'Marina Bay': 1, 'Melbourne': 2, 'Mexico City': 3, 'Miami': 1, 'Miami Gardens': 1, 'Monaco': 3, 'Montréal': 2, 'Monza': 1, 'Sakhir': 0, 'Shanghai': 0, 'Silverstone': 0, 'Spa-Francorchamps': 0, 'Spielberg': 3, 'Suzuka': 0, 'São Paulo': 2, 'Yas Island': 1, 'Zandvoort': 2}

Schedule loaded: 58 races across [2023, 2024, 2025]
Unknown clusters: 0

Cluster distribution:
circuit_cluster
0    15
1    19
2    10
3    14
Name: count, dtype: int64


---

## Step 2 — Load Race Laps, Track Status & Race Control Messages

For each of the 58 races (2023–2025) we load three data sources per session:

- **`session.laps`** — one row per driver per lap; aggregated to race-lap level for
  the base features (lap times, tyre life, retirements).
- **`session.track_status`** — timestamped DataFrame recording every track status
  change during the race at second-level resolution. Unlike the per-lap
  `TrackStatus` field on laps, this gives us the *exact moment* a yellow or SC
  was called, allowing us to count how many status changes occurred within each
  lap window.
- **`session.race_control_messages`** — timestamped FIA messages including incident
  reports, yellow flag sector notifications, and SC/VSC announcements. These are
  the observable precursor signals that precede most SC deployments by 1–2 laps.

`track_status` and `race_control_messages` are stored as per-race dicts keyed by
`"{year}_{round}"` and used in Step 3 to engineer the richer lap-window features.

### TrackStatus field

`TrackStatus` is a **concatenated string of single-digit FIA codes** — a lap that
crosses a status boundary carries both codes (e.g. `'21'` for yellow then clear,
`'46'` for SC entering VSC). The full code table:

| Code | Status | Race relevance |
|------|--------|----------------|
| `'1'` | Track clear — green flag | Normal racing |
| `'2'` | Yellow flag (local caution) | Incident in sector |
| `'4'` | **Safety Car deployed** | Full-course caution |
| `'5'` | Red flag — session stopped | Rare; aborts the race |
| `'6'` | **Virtual Safety Car** | Speed-delta caution |
| `'7'` | VSC ending (transition) | Brief before returning to green |

`most_severe_status` parses each character independently and returns the single
highest-severity code: `'4'` (SC) > `'6'` (VSC) > `'5'` (red) > `'7'` (VSC ending)
> `'2'` (yellow) > `'1'` (green).


In [4]:
# ── Step 2 — Load race laps + track_status + race_control_messages ────────────
# TrackStatus is a *concatenated* string of single-digit FIA codes (e.g. "12","46").
# Severity: SC(4) > VSC(6) > Red(5) > VSCending(7) > Yellow(2) > Green(1)
STATUS_SEVERITY = {"1": 0, "2": 1, "7": 2, "5": 3, "6": 4, "4": 5}

def most_severe_status(statuses: pd.Series) -> str:
    """Parse every character of every concatenated TrackStatus string and return
    the single code with the highest severity seen across all driver rows on that lap."""
    best_sev, best_code = -1, "1"
    for raw in statuses.dropna().astype(str):
        for ch in raw:
            sev = STATUS_SEVERITY.get(ch, -1)
            if sev > best_sev:
                best_sev, best_code = sev, ch
    return best_code

rows     = []
failed   = []
ts_store  = {}   # {race_id: track_status DataFrame}  — used in Step 3
rcm_store = {}   # {race_id: race_control_messages DataFrame} — used in Step 3

for _, race in schedule.iterrows():
    year       = int(race["year"])
    rnd        = int(race["RoundNumber"])
    event_name = race["EventName"]
    cluster    = int(race["circuit_cluster"])
    race_id    = f"{year}_{rnd}"

    try:
        session = fastf1.get_session(year, rnd, "R")
        session.load(laps=True, weather=True, telemetry=False, messages=True)

        laps = session.laps.copy()
        if laps.empty:
            failed.append((year, rnd, event_name, "empty laps"))
            continue

        # ── Store timestamped sources for Step 3 ──────────────────────────────
        try:
            ts_store[race_id] = session.track_status.copy()
        except Exception:
            ts_store[race_id] = pd.DataFrame(columns=["Time", "Status"])

        try:
            rcm_store[race_id] = session.race_control_messages.copy()
        except Exception:
            rcm_store[race_id] = pd.DataFrame(columns=["Time", "Category", "Message", "Flag", "Scope", "Sector"])

        # ── Per-lap aggregation ────────────────────────────────────────────────
        laps["TrackStatus"] = laps["TrackStatus"].astype(str)
        total_laps = laps["LapNumber"].max()
        grp = laps.groupby("LapNumber")

        lap_agg = grp.agg(
            lap_time_mean  = ("LapTime",     lambda x: x.dt.total_seconds().mean()),
            lap_time_std   = ("LapTime",     lambda x: x.dt.total_seconds().std()),
            lap_time_min   = ("LapTime",     lambda x: x.dt.total_seconds().min()),
            n_drivers      = ("Driver",      "count"),
            tyre_life_mean = ("TyreLife",    "mean"),
            tyre_life_max  = ("TyreLife",    "max"),
            track_status   = ("TrackStatus", most_severe_status),
        ).reset_index()

        # Retirements: cumulative count of drivers who disappeared from the field
        drivers_per_lap = grp["Driver"].apply(set)
        all_drivers     = set(laps["Driver"].unique())
        lap_agg["retirements"] = lap_agg["LapNumber"].apply(
            lambda n: len(all_drivers - set().union(*drivers_per_lap[:n]))
        )

        if "Rainfall" in laps.columns:
            rain = grp["Rainfall"].agg(lambda x: int(x.mode()[0]) if not x.mode().empty else 0)
            lap_agg["rainfall"] = lap_agg["LapNumber"].map(rain).fillna(0).astype(int)
        else:
            lap_agg["rainfall"] = 0

        lap_agg["race_id"]         = race_id
        lap_agg["year"]            = year
        lap_agg["round"]           = rnd
        lap_agg["event_name"]      = event_name
        lap_agg["circuit_cluster"] = cluster
        lap_agg["total_laps"]      = total_laps
        lap_agg["lap_pct"]         = lap_agg["LapNumber"] / total_laps

        rows.append(lap_agg)

    except Exception as e:
        failed.append((year, rnd, event_name, str(e)))
        print(f"  FAIL {year} R{rnd} {event_name}: {e}")

raw = pd.concat(rows, ignore_index=True)

print(f"Loaded      : {len(rows)} races  |  {len(raw)} lap-rows")
print(f"Failed      : {len(failed)}")
print(f"ts_store    : {len(ts_store)} entries  |  sample cols: {list(next(iter(ts_store.values())).columns)}")
print(f"rcm_store   : {len(rcm_store)} entries  |  sample cols: {list(next(iter(rcm_store.values())).columns)}")
print(f"\ntrack_status distribution:")
print(raw["track_status"].value_counts().sort_index())
print(raw[["race_id", "LapNumber", "track_status", "retirements"]].head(8))


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '14', '55', '44', '18', '63', '77', '10', '23', '22', '2', '20', '21', '27', '24', '4', '31', '16', '81']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached d

Loaded      : 58 races  |  3506 lap-rows
Failed      : 0
ts_store    : 58 entries  |  sample cols: ['Time', 'Status', 'Message']
rcm_store   : 58 entries  |  sample cols: ['Time', 'Category', 'Message', 'Status', 'Flag', 'Scope', 'Sector', 'RacingNumber', 'Lap']

track_status distribution:
track_status
1    3133
2     140
4     173
5       2
6      58
Name: count, dtype: int64
  race_id  LapNumber track_status  retirements
0  2023_1        1.0            2            0
1  2023_1        2.0            2            0
2  2023_1        3.0            1            0
3  2023_1        4.0            1            0
4  2023_1        5.0            1            0
5  2023_1        6.0            1            0
6  2023_1        7.0            1            0
7  2023_1        8.0            1            0


**Step 2 results — 58 races loaded, 0 failures, 3,506 lap-rows.**

Track status breakdown across all laps:

| Status | Code | Count | % laps |
|--------|------|-------|--------|
| Green  | `1`  | 3,133 | 89.4%  |
| Yellow | `2`  |   140 |  4.0%  |
| SC     | `4`  |   173 |  4.9%  |
| VSC    | `6`  |    58 |  1.7%  |
| Red    | `5`  |     2 |  0.1%  |

SC + VSC combined: **231 caution laps (6.6%)** — this is the raw caution rate at lap
level, before the 5-lap forward window in Step 4 inflates the positive class.
The 140 yellow-flag laps are candidate precursors; whether they systematically precede
SC deployments is exactly what the EDA in Step 5 will confirm.

Two useful structural findings from the loaded dicts:
- `ts_store` carries a `Message` column alongside `Time`/`Status`, providing
  human-readable context for each status change (e.g. `"SAFETY CAR DEPLOYED"`).
- `rcm_store` includes a `Lap` column — direct integer join by lap number in Step 3,
  no timestamp interpolation needed.
